# Checkpoint 11 — LightGBM evaluation

This checkpoint trains LightGBM against the same filtered horizon CSVs and frozen five-fold channel-grouped development splits used by XGBoost. The reserved test partition remains untouched. Lower WAPE and RMSLE are better; view-capture values closer to 100% are better.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
LIGHTGBM_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint11_lightgbm'
XGBOOST_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint10_wape_blended'
HORIZONS = (7, 14, 21, 30)
print(f'Project root: {PROJECT_ROOT}')
print(f'LightGBM checkpoint exists: {LIGHTGBM_DIR.exists()}')

Project root: D:\ViewCastLK
LightGBM checkpoint exists: True


## 1. Optional retraining

The checked-in code is reproducible, but the default is to reuse the completed checkpoint. Set `RUN_TRAINING = True` to repeat all 120 LightGBM fits.

In [2]:
RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run(
        [sys.executable, '-u', str(PROJECT_ROOT / 'scripts' / 'train_lightgbm_models.py')],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print('Using the completed checkpoint11_lightgbm artifacts.')

Using the completed checkpoint11_lightgbm artifacts.


## 2. Per-horizon comparison with the selected XGBoost ensembles

In [3]:
lightgbm_selected = pd.read_csv(LIGHTGBM_DIR / 'selected_models.csv')
xgboost_selected = pd.read_csv(XGBOOST_DIR / 'selected_ensembles.csv')
comparison = lightgbm_selected[[
    'horizon_days', 'candidate', 'wape_pct', 'rmsle',
    'total_view_capture_pct', 'top_decile_view_capture_pct'
]].merge(
    xgboost_selected[[
        'horizon_days', 'cross_fitted_oof_wape_pct',
        'cross_fitted_oof_rmsle',
        'cross_fitted_oof_total_view_capture_pct'
    ]],
    on='horizon_days',
).rename(columns={
    'wape_pct': 'lightgbm_wape_pct',
    'rmsle': 'lightgbm_rmsle',
    'total_view_capture_pct': 'lightgbm_view_capture_pct',
    'top_decile_view_capture_pct': 'lightgbm_top10_capture_pct',
    'cross_fitted_oof_wape_pct': 'xgboost_wape_pct',
    'cross_fitted_oof_rmsle': 'xgboost_rmsle',
    'cross_fitted_oof_total_view_capture_pct': 'xgboost_view_capture_pct',
})
comparison['wape_change_pp'] = comparison['lightgbm_wape_pct'] - comparison['xgboost_wape_pct']
display(comparison.round(3))

,horizon_days,candidate,lightgbm_wape_pct,lightgbm_rmsle,lightgbm_view_capture_pct,lightgbm_top10_capture_pct,xgboost_wape_pct,xgboost_rmsle,xgboost_view_capture_pct,wape_change_pp
0,7,log_l1_all_rows,92.173,2.004,20.868,9.904,92.954,1.987,17.778,-0.781
1,14,log_l2_all_rows,93.594,2.001,14.372,6.824,93.780,2.046,21.758,-0.186
2,21,log_l2_all_rows,89.738,2.157,24.056,15.921,92.650,2.176,25.590,-2.912
3,30,log_l1_all_rows,92.697,2.065,17.574,8.404,92.533,2.084,19.939,0.164


## 3. Combined development OOF evaluation

In [4]:
lightgbm_manifest = json.loads((LIGHTGBM_DIR / 'training_manifest.json').read_text())
xgboost_manifest = json.loads((XGBOOST_DIR / 'training_manifest.json').read_text())
metric_keys = [
    'wape_pct', 'rmsle', 'median_absolute_error_views', 'mae_views',
    'total_view_capture_pct', 'top_decile_view_capture_pct', 'log_r2'
]
combined = pd.DataFrame([
    {'model': 'XGBoost ensemble', **{key: xgboost_manifest['selected_oof_combined_metrics'][key] for key in metric_keys}},
    {'model': 'LightGBM selected', **{key: lightgbm_manifest['selected_oof_combined_metrics'][key] for key in metric_keys}},
])
display(combined.round(3))

,model,wape_pct,rmsle,median_absolute_error_views,mae_views,total_view_capture_pct,top_decile_view_capture_pct,log_r2
0,XGBoost ensemble,93.010,2.067,1318.937,18637.965,21.109,9.893,0.33
1,LightGBM selected,92.138,2.053,1188.021,18463.197,19.044,9.998,0.34


## 4. Out-of-fold sample predictions

These examples are spaced across each horizon's actual-view distribution. They are validation predictions, not fitted training-row predictions.

In [5]:
samples = pd.read_csv(LIGHTGBM_DIR / 'sample_predictions.csv')
display(samples.round({'predicted_views': 1, 'absolute_error_views': 1}))

,horizon_days,video_id,actual_views,predicted_views,absolute_error_views
0,7,0JJjVetdiiU,0.0,37.9,37.9
1,7,jR8rUnEOqiI,80.0,522.9,442.9
2,7,1LM--vRFJhY,258.0,1657.1,1399.1
3,7,CYC35JI51KI,690.0,36.0,654.0
4,7,DTaNvu1PzKw,1510.0,449.9,1060.1
5,7,9Am0bjluX54,3892.0,1198.0,2694.0
6,7,ZHS6uKPBYug,16225.0,1439.6,14785.4
7,7,dPwe3bwr2rk,7779604.0,10512.4,7769091.6
8,14,Nhdw8P3cNbQ,0.0,72.2,72.2
9,14,TbS2vW2W1D0,89.0,297.9,208.9


## 5. Highest-gain features by horizon

In [6]:
importance = pd.read_csv(LIGHTGBM_DIR / 'feature_importance.csv')
top_features = importance.sort_values(
    ['horizon_days', 'gain_share'], ascending=[True, False]
).groupby('horizon_days').head(10)
display(top_features[['horizon_days', 'feature', 'gain_share', 'split_count']].round(4))

,horizon_days,feature,gain_share,split_count
0,7,ch_avg_views_per_video_at_publish,0.2906,644
1,7,duration_seconds,0.1916,848
2,7,ch_subs_at_publish,0.1492,685
3,7,channel_age_days_at_publish,0.0808,613
4,7,ch_videos_at_publish,0.0738,449
5,7,ch_videos_per_day,0.0532,465
6,7,category_encoded_log,0.0286,231
7,7,topic_politics,0.0199,49
8,7,topic_entertainment,0.0095,65
9,7,topic_fashion,0.0092,53


## 6. Executable artifact checks

In [7]:
validation = pd.read_csv(LIGHTGBM_DIR / 'validation_tests.csv')
assert validation['status'].eq('PASS').all()
assert lightgbm_manifest['algorithm'] == 'LightGBM'
assert lightgbm_manifest['reserved_test_used'] is False
assert {record['horizon_days'] for record in lightgbm_manifest['models']} == set(HORIZONS)
for horizon in HORIZONS:
    model_path = LIGHTGBM_DIR / 'models' / f'day_{horizon}_lightgbm.joblib'
    bundle = joblib.load(model_path)
    assert bundle.regressor.__class__.__module__.startswith('lightgbm')
    frame = pd.read_csv(
        PROJECT_ROOT / 'Dataset' / 'model_horizon_datasets' / f'viewcastlk_day_{horizon}.csv',
        nrows=3,
    ).drop(columns=[f'd{horizon}_views'])
    prediction = bundle.predict_views(frame)
    assert len(prediction) == 3
    assert np.isfinite(prediction).all() and (prediction >= 0).all()
display(validation)
print('PASS: all four saved LightGBM models loaded and predicted successfully; reserved test remains untouched.')

,test,status
0,day 7 development OOF coverage,PASS
1,day 7 reserved test untouched,PASS
2,day 7 saved LightGBM smoke prediction,PASS
3,day 14 development OOF coverage,PASS
4,day 14 reserved test untouched,PASS
5,day 14 saved LightGBM smoke prediction,PASS
6,day 21 development OOF coverage,PASS
7,day 21 reserved test untouched,PASS
8,day 21 saved LightGBM smoke prediction,PASS
9,day 30 development OOF coverage,PASS


PASS: all four saved LightGBM models loaded and predicted successfully; reserved test remains untouched.
